# Window Size Sweep on GSM8K

扫描 `window_blocks` 0-4，对比两组策略 + 各自原版 baseline：

| 任务名 | 说明 | suffix_mode |
|--------|------|-------------|
| **expand_baseline** | expand + rewarm + fb（原版，不裁剪） | None |
| **expand_w0 ~ w4** | expand + rewarm + fb + window 裁剪 | window |
| **dualcache_baseline** | 纯 dual-cache（原版，不裁剪） | None |
| **dualcache_w0 ~ w4** | 纯 dual-cache + window 裁剪 | window |

共 12 个任务，seed=42 锁定。

**任务池 + GPU 池调度**

## 1. 环境设置

In [ ]:
import os
import torch
import gc

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} '
          f'({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)')

## 2. 配置 & 任务池 / GPU 池

In [ ]:
import subprocess
import datetime
import threading
import queue

SEED = 42

# ========== Sweep axis ==========
WINDOW_BLOCKS_RANGE = list(range(5))   # 0, 1, 2, 3, 4

MTR_VALUES = [0.7, 0.9]

TASK_CONFIGS = []
for mtr in MTR_VALUES:
    # --- Baseline: no suffix pruning ---
    TASK_CONFIGS.append({
        'group': f'mtr{mtr}', 'suffix_mode': None, 'window_blocks': 0,
        'dual_cache': True, 'mid_block_expand': True,
        'mid_trigger_ratio': mtr, 'rewarm_on_expand': True,
        'front_block_fallback_only': True,
    })
    # --- window_blocks 0-4 ---
    for wb in WINDOW_BLOCKS_RANGE:
        TASK_CONFIGS.append({
            'group': f'mtr{mtr}', 'suffix_mode': 'window', 'window_blocks': wb,
            'dual_cache': True, 'mid_block_expand': True,
            'mid_trigger_ratio': mtr, 'rewarm_on_expand': True,
            'front_block_fallback_only': True,
        })


def config_name(cfg):
    if cfg['suffix_mode'] is None:
        return f"{cfg['group']}_baseline"
    return f"{cfg['group']}_w{cfg['window_blocks']}"

ALL_GROUPS = sorted(set(c['group'] for c in TASK_CONFIGS))


# ========== GPU pool: 从 CUDA_VISIBLE_DEVICES 自动获取 ==========
GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]

# ========== Eval params ==========
task = 'gsm8k'
fewshot = 5
limit = None        # full GSM8K (1319 samples)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

gen_length = 256
steps = 256
block_length = 32
threshold = 0.9

print(f'Total tasks: {len(TASK_CONFIGS)}  (seed={SEED})')
for cfg in TASK_CONFIGS:
    sm = str(cfg['suffix_mode']) if cfg['suffix_mode'] else 'None'
    print(f'  {config_name(cfg):24s}  suffix_mode={sm:6s}  wb={cfg["window_blocks"]}')
print(f'GPU pool: {GPU_POOL} ({len(GPU_POOL)} GPUs)')
print(f'limit: {limit} ({"full" if limit is None else f"{limit} samples"})')
print(f'Timestamp: {timestamp}')

## 3. 启动任务池

In [ ]:
task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = config_name(cfg)
        log_file = f'nlogs/sweep_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/window_sweep/{task}-{name}-{timestamp}'
        records_dir = f'{output_dir}/step_records'

        base_args = [
            f"model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            f'block_length={block_length}',
            f'threshold={threshold}',
            'use_cache=True',
            'show_speed=True',
            f"step_records_dir='{records_dir}'",
            f'seed={SEED}',
        ]

        extra_args = [
            f"dual_cache={cfg['dual_cache']}",
            f"mid_block_expand={cfg['mid_block_expand']}",
            f"mid_trigger_ratio={cfg['mid_trigger_ratio']}",
            f"rewarm_on_expand={cfg['rewarm_on_expand']}",
            f"front_block_fallback_only={cfg['front_block_fallback_only']}",
        ]
        if cfg['suffix_mode'] is not None:
            extra_args.append(f"suffix_mode={cfg['suffix_mode']}")
            extra_args.append(f"window_blocks={cfg['window_blocks']}")

        model_args_str = ','.join(base_args + extra_args)

        limit_str = f' --limit {limit}' if limit is not None else ''
        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot}{limit_str} '
            f'--confirm_run_unsafe_code --model llada_dist '
            f'--model_args {model_args_str} '
            f'--output_path {output_dir} --log_samples'
        )

        print(f'[GPU {gpu_id}] START  {name}')

        p = subprocess.Popen(
            cmd, shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()

        status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
        print(f'[GPU {gpu_id}] DONE   {name}  {status}')

        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))

        task_queue.task_done()


threads = []
for gpu_id in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gpu_id,), daemon=True)
    t.start()
    threads.append(t)

print(f'\nLaunched {len(threads)} GPU workers for {len(TASK_CONFIGS)} tasks.')
print('Waiting for all tasks to complete...')

In [ ]:
for t in threads:
    t.join()

print(f'\nAll {len(all_results)} / {len(TASK_CONFIGS)} tasks finished.')
for cfg, name, log_file, output_dir, rc in all_results:
    status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
    print(f'  {name:20s}  {status}  log={log_file}')

## 4. 解析评测结果

In [ ]:
import re
import json
import pandas as pd

parsed_results = []
for cfg in TASK_CONFIGS:
    name = config_name(cfg)
    log_file = f'nlogs/sweep_{task}_{name}_{timestamp}.log'
    if not os.path.exists(log_file):
        print(f'WARNING: {log_file} not found')
        continue
    with open(log_file, 'r') as f:
        content = f.read()

    acc_match = re.search(r'exact_match.*?[\|,]\s*[\|]?\s*([\d.]+)', content)
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)

    parsed_results.append({
        'group': cfg['group'],
        'suffix_mode': cfg['suffix_mode'],
        'window_blocks': cfg['window_blocks'],
        'config': name,
        'is_baseline': cfg['suffix_mode'] is None,
        'accuracy': float(acc_match.group(1)) if acc_match else None,
        'tokens_per_sec': float(speed_match.group(1)) if speed_match else None,
        'total_nfe': int(nfe_match.group(1)) if nfe_match else None,
        'time_sec': float(time_match.group(1)) if time_match else None,
        'log_file': log_file,
    })

df = pd.DataFrame(parsed_results)
show_cols = ['config', 'accuracy', 'total_nfe', 'time_sec', 'tokens_per_sec']
print('=== expand (rewarm + fb) ===')
display(df[df['group'] == 'expand'][show_cols].reset_index(drop=True))
print('\n=== dualcache ===')
display(df[df['group'] == 'dualcache'][show_cols].reset_index(drop=True))

## 5. 加载 Step Records

In [ ]:
step_data = {}
for cfg in TASK_CONFIGS:
    name = config_name(cfg)
    rpath = f'evals_results/window_sweep/{task}-{name}-{timestamp}/step_records/step_records.json'
    if os.path.exists(rpath):
        with open(rpath, 'r') as f:
            step_data[name] = json.load(f)
        print(f'  [OK]   {name}: {len(step_data[name])} samples')
    else:
        print(f'  [MISS] {name}')

print(f'\nLoaded step records for {len(step_data)} / {len(TASK_CONFIGS)} configs.')
print(f'  Baselines: {[config_name(c) for c in TASK_CONFIGS if c["suffix_mode"] is None]}')

## 6. Accuracy / NFE / Speed 对比图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

GROUP_COLORS = {g: c for g, c in zip(ALL_GROUPS, ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0'])}
METRICS = [
    ('accuracy', 'GSM8K Accuracy', 'Accuracy'),
    ('total_nfe', 'Total NFE', 'NFE'),
    ('tokens_per_sec', 'Tokens / sec', 'Speed'),
]

df_valid = df.dropna(subset=['accuracy']).copy()
df_baseline = df_valid[df_valid['is_baseline'] == True]
df_window = df_valid[df_valid['is_baseline'] == False]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for group_name, color in GROUP_COLORS.items():
    # --- baseline: horizontal dashed line ---
    bl = df_baseline[df_baseline['group'] == group_name]
    if not bl.empty:
        bl_row = bl.iloc[0]
        for ax_idx, (col, _, _) in enumerate(METRICS):
            val = bl_row.get(col)
            if pd.notna(val):
                axes[ax_idx].axhline(y=val, color=color, linestyle='--', alpha=0.6,
                                     label=f'{group_name}_baseline ({val:.4f})'
                                           if col == 'accuracy' else f'{group_name}_baseline')

    # --- window sweep: solid line ---
    sub = df_window[df_window['group'] == group_name].sort_values('window_blocks')
    if sub.empty:
        continue
    wb = sub['window_blocks'].values

    for ax_idx, (col, ylabel, title) in enumerate(METRICS):
        vals = sub[col].values
        if pd.isna(vals).all():
            continue
        axes[ax_idx].plot(wb, vals, 'o-', color=color,
                          label=f'{group_name}_window', linewidth=2, markersize=8)
        for w, v in zip(wb, vals):
            if pd.notna(v):
                fmt = f'{v:.4f}' if col == 'accuracy' else (str(int(v)) if col == 'total_nfe' else f'{v:.1f}')
                axes[ax_idx].annotate(fmt, (w, v), textcoords='offset points',
                                      xytext=(0, 10), ha='center', fontsize=8, color=color)

for ax_idx, (col, ylabel, title) in enumerate(METRICS):
    axes[ax_idx].set_xlabel('window_blocks')
    axes[ax_idx].set_xticks(WINDOW_BLOCKS_RANGE)
    axes[ax_idx].set_ylabel(ylabel)
    axes[ax_idx].set_title(f'{title} vs window_blocks', fontweight='bold')
    axes[ax_idx].legend(fontsize=8)
    axes[ax_idx].grid(True, alpha=0.3)

fig.suptitle('Window Size Sweep (GSM8K, expand vs dualcache, seed=42)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../eval_results', exist_ok=True)
plt.savefig('../eval_results/window_sweep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_results/window_sweep_comparison.png')

## 7. Per-Group Step 分布 (expand 组)

In [ ]:
from collections import defaultdict

def plot_step_dist(group_name, configs, step_data, parsed_results):
    available = [config_name(c) for c in configs if config_name(c) in step_data]
    if not available:
        print(f'No step_data for {group_name}')
        return

    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 5), sharey=True)
    if n == 1:
        axes = [axes]

    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, n))

    for ax_idx, cname in enumerate(available):
        ax = axes[ax_idx]
        samples = step_data[cname]

        step_transferred = defaultdict(list)
        for sample_records in samples:
            for rec in sample_records:
                step_transferred[rec['global_step']].append(rec['transferred'])

        max_step = max(step_transferred.keys()) if step_transferred else 0
        steps_range = list(range(max_step + 1))
        means = [np.mean(step_transferred[s]) if s in step_transferred else 0 for s in steps_range]

        ax.bar(steps_range, means, color=cmap[ax_idx], alpha=0.7)
        ax.set_xlabel('Global Step')

        res = next((r for r in parsed_results if r['config'] == cname), None)
        acc_s = f"Acc={res['accuracy']:.4f}" if res and res['accuracy'] else ''
        nfe_s = f"NFE={res['total_nfe']}" if res and res['total_nfe'] else ''
        label = cname.replace(f'{group_name}_', '')
        ax.set_title(f'{label}\n{acc_s}  {nfe_s}', fontsize=10)
        ax.set_xlim(-0.5, max_step + 0.5)

    axes[0].set_ylabel('Tokens Transferred')
    fig.suptitle(f'Per-Step Decoded Tokens ({group_name})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out_path = f'../eval_results/window_sweep_steps_{group_name}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')


mtr07_cfgs = [c for c in TASK_CONFIGS if c['group'] == 'mtr0.7']
plot_step_dist('mtr0.7', mtr07_cfgs, step_data, parsed_results)

## 8. Per-Group Step 分布 (mtr0.9 组)

In [ ]:
mtr09_cfgs = [c for c in TASK_CONFIGS if c['group'] == 'mtr0.9']
plot_step_dist('mtr0.9', mtr09_cfgs, step_data, parsed_results)

## 9. 从已有结果重新加载

In [ ]:
import glob, re, json, os
import pandas as pd

task = 'gsm8k'
SEED = 42
WINDOW_BLOCKS_RANGE = list(range(5))

MTR_VALUES = [0.7, 0.9]
TASK_CONFIGS = []
for mtr in MTR_VALUES:
    TASK_CONFIGS.append({'group': f'mtr{mtr}', 'suffix_mode': None, 'window_blocks': 0,
                         'dual_cache': True, 'mid_block_expand': True, 'mid_trigger_ratio': mtr,
                         'rewarm_on_expand': True, 'front_block_fallback_only': True})
    for wb in WINDOW_BLOCKS_RANGE:
        TASK_CONFIGS.append({'group': f'mtr{mtr}', 'suffix_mode': 'window', 'window_blocks': wb,
                             'dual_cache': True, 'mid_block_expand': True, 'mid_trigger_ratio': mtr,
                             'rewarm_on_expand': True, 'front_block_fallback_only': True})
ALL_GROUPS = sorted(set(c['group'] for c in TASK_CONFIGS))

def config_name(cfg):
    if cfg['suffix_mode'] is None:
        return f"{cfg['group']}_baseline"
    return f"{cfg['group']}_w{cfg['window_blocks']}"

first_name = config_name(TASK_CONFIGS[0])
latest_logs = sorted(
    glob.glob(f'nlogs/sweep_{task}_{first_name}_*.log'),
    key=os.path.getmtime, reverse=True,
)
if latest_logs:
    fname = os.path.basename(latest_logs[0])
    timestamp = fname.replace(f'sweep_{task}_{first_name}_', '').replace('.log', '')
    print(f'Auto-detected latest timestamp: {timestamp}')
else:
    timestamp = 'NOTFOUND'
    print('WARNING: No log found!')

step_data = {}
for cfg in TASK_CONFIGS:
    name = config_name(cfg)
    rpath = f'evals_results/window_sweep/{task}-{name}-{timestamp}/step_records/step_records.json'
    if os.path.exists(rpath):
        with open(rpath, 'r') as f:
            step_data[name] = json.load(f)
        print(f'  [OK]   {name}: {len(step_data[name])} samples')
    else:
        print(f'  [MISS] {name}')

parsed_results = []
for cfg in TASK_CONFIGS:
    name = config_name(cfg)
    log_file = f'nlogs/sweep_{task}_{name}_{timestamp}.log'
    if not os.path.exists(log_file):
        continue
    with open(log_file, 'r') as f:
        content = f.read()

    acc_match = re.search(r'exact_match.*?[\|,]\s*[\|]?\s*([\d.]+)', content)
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    token_match = re.search(r'Total number of tokens generated:\s*(\d+)', content)
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)

    parsed_results.append({
        'group': cfg['group'],
        'suffix_mode': cfg['suffix_mode'],
        'window_blocks': cfg['window_blocks'],
        'config': name,
        'is_baseline': cfg['suffix_mode'] is None,
        'accuracy': float(acc_match.group(1)) if acc_match else None,
        'total_nfe': int(nfe_match.group(1)) if nfe_match else None,
        'time_sec': float(time_match.group(1)) if time_match else None,
        'total_tokens': int(token_match.group(1)) if token_match else None,
        'tokens_per_sec': float(speed_match.group(1)) if speed_match else None,
    })

df = pd.DataFrame(parsed_results)
print(f'Loaded {len(step_data)} step_data, {len(parsed_results)} parsed_results  (seed={SEED})\n')

# Tab-separated output — paste directly into Excel / Google Sheets
header = '\t'.join(['Config', 'Group', 'SuffixMode', 'WB', 'ACC', 'NFE', 'Time(s)', 'Total Token', 'Token/s', 'StepRec'])
print(header)
for r in parsed_results:
    row = '\t'.join([
        r['config'],
        r['group'],
        str(r['suffix_mode']) if r['suffix_mode'] else 'None',
        str(r['window_blocks']),
        f"{r['accuracy']:.4f}" if r['accuracy'] is not None else '',
        str(r['total_nfe']) if r['total_nfe'] is not None else '',
        f"{r['time_sec']:.1f}" if r['time_sec'] is not None else '',
        str(r['total_tokens']) if r['total_tokens'] is not None else '',
        f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] is not None else '',
        'YES' if r['config'] in step_data else 'NO',
    ])
    print(row)

print(f'\nRe-run Section 6~8 cells to regenerate plots.')